# LightGBM for Malware Detection

This notebook implements LightGBM (Light Gradient Boosting Machine) for binary classification of malware vs goodware.

LightGBM is a fast, distributed, high-performance gradient boosting framework that uses tree-based learning algorithms.

## Steps:
1. Load and explore the dataset
2. Preprocess the data
3. Split into train/test sets (80/20)
4. Train LightGBM model
5. Evaluate using 10-fold cross-validation
6. Test on hold-out test set
7. Analyze feature importance
8. Save the model

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report
import lightgbm as lgb
import pickle
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Load and Explore Dataset

In [ ]:
# Load the dataset
df = pd.read_csv('../dataset/brazilian-malware-dataset/brazilian-malware.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Check class distribution
print("Class distribution:")
print(df['Label'].value_counts())
print(f"\nClass balance: {df['Label'].value_counts(normalize=True)}")

## 2. Data Preprocessing

In [ ]:
# Separate features and target
X = df.drop('Label', axis=1)

# Keep only numeric columns
numeric_cols = X.select_dtypes(include=[np.number]).columns
X = X[numeric_cols]

y = df['Label']

# Handle missing values
if X.isnull().sum().sum() > 0:
    print("Filling missing values with 0...")
    X = X.fillna(0)

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

## 3. Train/Test Split (80/20)

In [ ]:
# Split into train and test sets (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")

## 4. Feature Scaling

Note: LightGBM doesn't require scaling, but we'll do it for consistency.

In [ ]:
# Initialize and fit scaler on training data only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features scaled successfully.")

## 5. Train LightGBM Model

In [ ]:
# Initialize LightGBM model with optimized hyperparameters
lgb_model = lgb.LGBMClassifier(
    n_estimators=100,        # Number of boosting rounds
    max_depth=6,             # Maximum tree depth
    learning_rate=0.1,       # Step size shrinkage
    num_leaves=31,           # Maximum number of leaves in one tree
    subsample=0.8,           # Subsample ratio of training instances
    colsample_bytree=0.8,    # Subsample ratio of columns
    random_state=RANDOM_STATE,
    n_jobs=-1,               # Use all CPU cores
    verbose=-1               # Suppress warnings
)

# Train the model
print("Training LightGBM model...")
lgb_model.fit(X_train_scaled, y_train)
print("Model training complete!")

## 6. Cross-Validation Evaluation (10-Fold)

In [ ]:
# Perform 10-fold stratified cross-validation
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)

print("Performing 10-fold cross-validation...")
print("This may take a few minutes...")
cv_results = cross_validate(
    lgb_model, 
    X_train_scaled, 
    y_train,
    cv=cv,
    scoring=['roc_auc', 'accuracy'],
    n_jobs=-1,
    return_train_score=False
)

# Display results
print("\n=== Cross-Validation Results ===")
print(f"AUC: {cv_results['test_roc_auc'].mean():.4f} ± {cv_results['test_roc_auc'].std():.4f}")
print(f"Accuracy: {cv_results['test_accuracy'].mean():.4f} ± {cv_results['test_accuracy'].std():.4f}")

## 7. Final Evaluation on Hold-Out Test Set

In [ ]:
# Make predictions on test set
y_pred = lgb_model.predict(X_test_scaled)
y_pred_proba = lgb_model.predict_proba(X_test_scaled)[:, 1]

# Calculate metrics
test_accuracy = accuracy_score(y_test, y_pred)
test_auc = roc_auc_score(y_test, y_pred_proba)

print("\n=== Test Set Results ===")
print(f"Accuracy: {test_accuracy:.4f}")
print(f"AUC: {test_auc:.4f}")

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)

# Visualize confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Goodware', 'Malware'],
            yticklabels=['Goodware', 'Malware'])
plt.title('Confusion Matrix - LightGBM')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

In [ ]:
# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Goodware', 'Malware']))

## 8. Feature Importance Analysis

In [ ]:
# Get feature importances
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': lgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 Most Important Features:")
print(feature_importance.head(10))

# Visualize top features
plt.figure(figsize=(10, 6))
top_features = feature_importance.head(15)
plt.barh(range(len(top_features)), top_features['importance'])
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Importance')
plt.title('Top 15 Feature Importances - LightGBM')
plt.tight_layout()
plt.show()

## 9. LightGBM-Specific Visualizations

In [ ]:
# Plot feature importance using LightGBM's built-in method
fig, ax = plt.subplots(figsize=(10, 8))
lgb.plot_importance(lgb_model, max_num_features=15, ax=ax, importance_type='gain')
plt.title('Feature Importance (Gain) - LightGBM')
plt.tight_layout()
plt.show()

## 10. Model Parameters

In [ ]:
# Display model parameters
print("Model Hyperparameters:")
print(f"Number of estimators: {lgb_model.n_estimators}")
print(f"Max depth: {lgb_model.max_depth}")
print(f"Learning rate: {lgb_model.learning_rate}")
print(f"Number of leaves: {lgb_model.num_leaves}")
print(f"Subsample: {lgb_model.subsample}")
print(f"Colsample by tree: {lgb_model.colsample_bytree}")

## 11. Save Model

In [ ]:
# Create models directory if it doesn't exist
import os
os.makedirs('../models', exist_ok=True)

# Save the model
with open('../models/lightgbm.pkl', 'wb') as f:
    pickle.dump(lgb_model, f)
print("Model saved to ../models/lightgbm.pkl")

## Summary

This notebook demonstrated:
1. ✅ Loading and exploring the malware dataset
2. ✅ Preprocessing with proper train/test split
3. ✅ Training a LightGBM classifier
4. ✅ 10-fold stratified cross-validation
5. ✅ Evaluation on hold-out test set
6. ✅ Feature importance analysis
7. ✅ LightGBM-specific visualizations
8. ✅ Model serialization for production use

**Key Results:**
- Cross-Validation AUC: [See results above]
- Test Set AUC: [See results above]
- Test Set Accuracy: [See results above]

**Advantages of LightGBM:**
- **Faster training speed** compared to XGBoost
- **Lower memory usage** - uses histogram-based algorithms
- **Better accuracy** on large datasets
- **Supports categorical features** natively
- **Handles large-scale data** efficiently
- **GPU support** for even faster training

**Key Features:**
- Leaf-wise tree growth (vs level-wise in XGBoost)
- Gradient-based One-Side Sampling (GOSS)
- Exclusive Feature Bundling (EFB)
- Histogram-based decision tree algorithm

**LightGBM vs XGBoost:**
- LightGBM is generally faster and uses less memory
- LightGBM grows trees leaf-wise (can be more accurate but prone to overfitting)
- XGBoost grows trees level-wise (more conservative)
- Both achieve similar accuracy on most datasets

**When to Use LightGBM:**
- Large datasets (>10K rows)
- Need for fast training
- Memory constraints
- Categorical features present